# New Experiments using Datasetv2

- First run, no imputation, no standardization. 
- coarse grid search

In [ ]:

import importlib
import feature_engineering as fe_module
import model as model_module
importlib.reload(fe_module)
importlib.reload(model_module)
from model import train_best_model
import os
import pandas as pd

files = [
    ("datasets/Dataset_v2/pooled_CN.csv", "CN"),
    ("datasets/Dataset_v2/pooled_MCI_AD.csv", "AD"),
]

param_grid = {
    'n_estimators': [100, 300, 500],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.001, 0.01, 0.05],
    'subsample': [0.4, 0.6, 0.8],
    'colsample_bytree': [0.4, 0.6, 0.8],
}

results_dir = "experiments/experiment_0.1/grid_results"
os.makedirs(results_dir, exist_ok=True)

exp0_1_results = {}
exp0_1_models = {}    # { key: (model, cols, summary) }

for path, prog in files:
    try:
        df = pd.read_csv(path)
        base = os.path.splitext(os.path.basename(path))[0]
        csv_out = os.path.join(results_dir, f"{base}_{prog}_cv_scores.csv")
        print(f"\n{'='*60}")
        print(f"=== Grid search: {base} ({prog}) — {len(df)} samples ===")
        print(f"{'='*60}")
        model, cols, summary = train_best_model(
            df,
            progression_type=prog,
            param_grid=param_grid,
            csv_path=csv_out,
            save_dir="experiments/experiment_0.1",
            cv_method='skf',
            n_jobs=8,
            model_base_name=base,
            save_artifacts=True,
        )
        key = f"{base}_{prog}"
        exp0_1_results[key] = pd.read_csv(csv_out)
        exp0_1_models[key]  = (model, cols, summary)
    except Exception as e:
        import traceback
        print(f"Error processing {path}: {e}")
        traceback.print_exc()



=== Grid search: pooled_CN (CN) — 12092 samples ===
Using StratifiedKFold with n_splits=5
Grid search: 2 hyperparameter combinations (n_jobs=8)


SKF grid search: 100%|██████████| 2/2 [00:14<00:00,  7.10s/combo]


Error processing datasets/Dataset_v2/pooled_CN.csv: Feature shape mismatch, expected: 301, got 303

=== Grid search: pooled_MCI_AD (AD) — 2891 samples ===


Traceback (most recent call last):
  File "/var/folders/j5/6cc1s4850711ycqjnnzgj1j00000gp/T/ipykernel_25217/1255299447.py", line 34, in <module>
    model, cols = train_best_model(
                  ~~~~~~~~~~~~~~~~^
        df,
        ^^^
    ...<7 lines>...
        save_artifacts=True,
        ^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/model.py", line 658, in train_best_model
    model, columns, imputer, scaler, summary = build_model_final(X_train, X_test, y_train, y_test, model_dict, feature_names)
                                               ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/model.py", line 233, in build_model_final
    model, imputer, scaler, y_pred, y_proba = _fit_xgb(X_train, X_test, y_train, model_dict)
                                              ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aeg00

Using StratifiedKFold with n_splits=5
Grid search: 2 hyperparameter combinations (n_jobs=8)


SKF grid search: 100%|██████████| 2/2 [00:05<00:00,  2.52s/combo]


Error processing datasets/Dataset_v2/pooled_MCI_AD.csv: Feature shape mismatch, expected: 301, got 303


Traceback (most recent call last):
  File "/var/folders/j5/6cc1s4850711ycqjnnzgj1j00000gp/T/ipykernel_25217/1255299447.py", line 34, in <module>
    model, cols = train_best_model(
                  ~~~~~~~~~~~~~~~~^
        df,
        ^^^
    ...<7 lines>...
        save_artifacts=True,
        ^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/model.py", line 658, in train_best_model
    model, columns, imputer, scaler, summary = build_model_final(X_train, X_test, y_train, y_test, model_dict, feature_names)
                                               ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aeg00011/Desktop/AD-Research/AD-Early-Prediction/model.py", line 233, in build_model_final
    model, imputer, scaler, y_pred, y_proba = _fit_xgb(X_train, X_test, y_train, model_dict)
                                              ~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/aeg00

In [ ]:

import importlib
import visualization as viz_module
importlib.reload(viz_module)
from visualization import plot_feature_importance, plot_confusion_mat, plot_roc
import os

charts_dir = "experiments/experiment_0.1/charts"
os.makedirs(charts_dir, exist_ok=True)

for key, (model, cols, summary) in exp0_1_models.items():
    print(f"\n{'='*60}")
    print(f"Visualizations — {key}")
    print(f"{'='*60}")

    # Feature importance
    plot_feature_importance(
        model.feature_importances_,
        cols,
        top_n=20,
        title=f"Top 20 Feature Importances — {key}",
        save_path=os.path.join(charts_dir, f"{key}_feature_importance.png"),
    )

    # Confusion matrix
    plot_confusion_mat(
        summary["y_true"],
        summary["y_pred"],
        title=f"Confusion Matrix — {key}",
        save_path=os.path.join(charts_dir, f"{key}_confusion_matrix.png"),
    )

    # ROC curve
    plot_roc(
        summary["y_true"],
        summary["y_proba"],
        title=f"ROC Curve — {key}",
        save_path=os.path.join(charts_dir, f"{key}_roc_curve.png"),
    )
